# {Project Title}📝

![Banner](./assets/banner.jpeg)

## Topic
*What problem are you (or your stakeholder) trying to address?*
📝 <!-- Answer Below -->

Historically, women have been underrepresented when it comes to health research and information. Female and male bodies are biologically different and can behave differently under the same circumstances or conditions, so it is very important that adequate research is done on both sexes so both halves of the population are able to be diagnosed and cared for correctly. There have been circumstances where female patients who are actively experiencing heart attacks are misdiagnosed or sent home because the common symptoms associated with it were drawn from research on men while women show different warning signs. 

I want to investigate the ratio of male and female representation in clincial trials overtime and see how it matches with the ratio of each sex experiencing that condition (their burden ratio). 

## Project Question
*What specific question are you seeking to answer with this project?*
*This is not the same as the questions you ask to limit the scope of the project.*
📝 <!-- Answer Below -->

Is there a gap in sex representation in clinical trials compared to their reported burden of the condition? Has this changed overtime?

Are there specific conditions that are more overlooked than others?

Is there a correlation between better representation and higher funding for different conditions?

## What would an answer look like?
*What is your hypothesized answer to your question?*
📝 <!-- Answer Below -->

The answer will be the difference between the ratio of male and female representation in trials and the ratio of male and female burden of that condition. The difference can be calculated initially by condition, then by year, and then all together to show how that has evolved, if at all, overtime. 

Each step in calculations will answer different parts of our overarching question. The first step will reveal if there are specific conditions that are more overlooked than others, then how much of a gap there is per year, and finally if a gap exists, how it has changed through the years. 

## Data Sources
*What 3 data sources have you identified for this project?*
*How are you going to relate these datasets?*
📝 <!-- Answer Below -->

**AACT** - *Aggregate Analysis of ClinicalTrials.gov*
Contains data elements for every study registered in ClinicalTrials.gov. Includes dates, trial phase data, participant counts by sex, and the conditions being studied.

**GBD** - *Global Burden of Disease*
The Institute for Health Metrics and Evaluation (IHME) publishes a searchable and downloadable set of data relating to GBD estimates. Covers from 1990 onward.

**NHAMCS** - *National Hospital Ambulatory Medical Care Survey*
Public CDC dataset of emergency department vists with information like patient sex, reason for visit, wait time, tests ordered, diagnosis, and whether patient was admitted or sent home.

The biggest difficulty will be connecting these datasets by condition. Conditions are defined differently and may need to be generalized with terms like "cardiovascular diseases". The joining table may need to be manually created to create connections to specific terms depending on how it is defined in each set. In addition to the condition, tables will be joined by date (year). This is how I will relate the AACT and GBD tables.

The NHAMCS will be analyzed based off of results of the other datasets. Where there tend to be gaps it can reveal more insight into how certain symptoms and conditions are treated between sexes. 


## Approach and Analysis
*What is your approach to answering your project question?*
*How will you use the identified data to answer your project question?*
📝 <!-- Start Discussing the project here; you can add as many code cells as you need -->

I will download the datasets from their sources and build the connector between the AACT and GBD datasets. These can be unified via the `mesh_term` and `cause_name` fields from each database respectively.

I will pull trials from the AACT dataset that have sex specific enrollment data and map total female and overall participants by condition and time period. Using the GBD dataset I can calculate how each sex has historically fared with a certain condition. By calculating something like the number of deaths, per case, per sex we can see how it is affecting each individually. Based off of each dataset we can also calculate the ratio of female research participants with their share of burden.

I can plot these overtime by condition and find if there are any significant gaps that correlate with a lack of research.

### AACT
Connected by establishing connection to database with credentials generated after creating account.

### GBD
Located in `data/` folder, downloaded from filterable queries and download abilities directly from site ([vizhub.healthdata.org/gbd-results](vizhub.healthdata.org/gbd-results)).

#### Filters
| Filter | Selection |
|---|---|
| GBD Estimate | Cause of death or injury |
| Measure | Deaths, DALYs, Prevalence, Incidence |
| Metric | Number, Rate |
| Cause | All except injuries and maternal conditions (since the latter only affects women) |
| Location | Global |
| Age | All ages, Age-standardized |
| Sex | Male, Female (not Both, each selected separately) |
| Total percentage change | False |
| Year | 2008–2022 |

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL, text
import pandas as pd

load_dotenv()

# -- AACT
# Establish connection
url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("AACT_USER"),
    password=os.getenv("AACT_PASSWORD"),
    host="aact-db.ctti-clinicaltrials.org",
    port=5432,
    database="aact",
)
engine = create_engine(url)

with engine.connect() as conn:
    df = pd.read_sql(text("SELECT nct_id, brief_title FROM studies LIMIT 5"), conn)
df

,nct_id,brief_title
0,NCT05917496,Analysis of Parental Support in Families Using...
1,NCT05945641,Effect of Low-load Resistance Training vs. Hig...
2,NCT02688673,DC Vaccine Combined With CIK Cells in Patients...
3,NCT00332839,Comparison of CNI-based Regimen Versus CNI-fre...
4,NCT00638560,Changes of Biomarkers in Response to Training ...


### NHAMCS
Downloaded from CDC's FTP server. Added code here so downloading is easily reproducible for others.

In [8]:
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile

BASE_URL = "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NHAMCS/stata/"
FILES = {
    2016: "ED2016-stata.zip", 2017: "ed2017-stata.zip", 2018: "ED2018-stata.zip",
    2019: "ED2019-stata.zip", 2020: "ed2020-stata.zip", 2021: "ed2021-stata.zip",
    2022: "ed2022-stata.zip",
}

data_dir = Path("data/nhamcs")
data_dir.mkdir(parents=True, exist_ok=True)

for year, filename in FILES.items():
    zip_path = data_dir / filename
    if not zip_path.exists():
        req = Request(BASE_URL + filename, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(req) as response, open(zip_path, "wb") as f:
            f.write(response.read())
        print(f"Downloaded {filename}")
    with ZipFile(zip_path) as z:
        z.extractall(data_dir / str(year))

Downloaded ED2016-stata.zip
Downloaded ed2017-stata.zip
Downloaded ED2018-stata.zip
Downloaded ED2019-stata.zip
Downloaded ed2020-stata.zip
Downloaded ed2021-stata.zip
Downloaded ed2022-stata.zip


## Resources and References
*What resources and references have you used for this project?*
📝 <!-- Answer Below -->

In [9]:
# ⚠️ Make sure you run this cell at the end of your notebook before every submission!
!jupyter nbconvert --to python source.ipynb

[NbConvertApp] Converting notebook source.ipynb to python
[NbConvertApp] Writing 6593 bytes to source.py
